# RNN - spam and ham dataset

In [1]:
# pip install tensorflow pandas scikit-learn pytz


In [2]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense


2025-08-19 16:53:14.436092: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-19 16:53:14.473162: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-08-19 16:53:15.430421: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
# df = pd.DataFrame(data)
df = pd.read_csv("./data/spam.csv")
df

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [4]:

# # -----------------------------
# # 0) Data: use existing df or sample
# # -----------------------------
# try:
#     assert 'df' in globals()
#     assert {'text','label'}.issubset(df.columns)
#     print("Using provided DataFrame `df`.")
# except:
#     print("No `df` found — using a small sample dataset.")
#     df = pd.DataFrame({
#         "text": [
#             "I love this movie",
#             "Worst film ever",
#             "Not bad could be better",
#             "Absolutely fantastic",
#             "Terrible acting and boring",
#             "I really enjoyed it",
#         ],
#         "label": [1, 0, 1, 1, 0, 1]
#     })


In [5]:
df.columns

Index(['Category', 'Message'], dtype='object')

In [6]:

# -----------------------------
# 1) Clean text (simple)
# -----------------------------
def clean_text(s):
    s = s.lower()
    s = re.sub(r"[^a-z\s']", " ", s) # keep letters/apostrophes
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["Message"] = df["Message"].astype(str).apply(clean_text)
# df["Liked"] = df["Liked"].astype(int)

# string labels to integers manually using map()
# df["Category"] = df["Category"].map({"ham": 0, "spam": 1})

# Alternate method to handle string labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["Category"] = le.fit_transform(df["Category"])
# ham -> 0, spam -> 1  (order is learned from sorted classes)

print('Category', df["Category"].unique() )
print("Classes learned by LabelEncoder:", le.classes_)
print("Mapping (class → integer):")
for cls, val in zip(le.classes_, range(len(le.classes_))):
    print(f"{cls} → {val}")

# Verify with first few rows
print(df[["Category"]].head().to_string(index=False))


X_train, X_test, y_train, y_test = train_test_split(
    df["Message"], df["Category"], test_size=0.33, random_state=42, stratify=df["Category"]
)


Category [0 1]
Classes learned by LabelEncoder: ['ham' 'spam']
Mapping (class → integer):
ham → 0
spam → 1
 Category
        0
        0
        1
        0
        0


In [7]:

# -----------------------------
# 2) Tokenize + PAD (many-to-one requires fixed length for batching)
# -----------------------------
MAX_VOCAB = 10000 # keep top words

tok = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tok.fit_on_texts(X_train)

train_seqs = tok.texts_to_sequences(X_train)
MAX_LEN = int(np.percentile([len(s) for s in train_seqs], 95))
# MAX_LEN = 20 # pad / truncate to this length

print('MAX_LEN', MAX_LEN)

def to_padded(texts):
    seqs = tok.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LEN, padding="post", truncating="post", value=0)

Xtr = to_padded(X_train)
Xte = to_padded(X_test)
ytr = y_train.values
yte = y_test.values

print("\nSample sequences BEFORE padding:", tok.texts_to_sequences(X_train[:2]))
print("Sample sequences AFTER padding:\n", Xtr[:2])
print("Each row length:", Xtr.shape[1], "(= MAX_LEN)")


MAX_LEN 32

Sample sequences BEFORE padding: [[490, 62, 36, 12, 64, 21, 58, 82, 79], [7, 960, 1691, 358, 3, 370, 3073, 168, 16, 3, 124, 188, 301, 640, 1239, 193, 543]]
Sample sequences AFTER padding:
 [[ 490   62   36   12   64   21   58   82   79    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0]
 [   7  960 1691  358    3  370 3073  168   16    3  124  188  301  640
  1239  193  543    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0]]
Each row length: 32 (= MAX_LEN)


In [8]:

# -----------------------------
# 3) Simple RNN (many-to-one)
# Embedding: converts token IDs to vectors
# SimpleRNN: returns FINAL hidden state (sequence summary)
# -----------------------------
EMBED_DIM = 32
RNN_UNITS = 32

model = Sequential([
    Embedding(
        input_dim=min(MAX_VOCAB, len(tok.word_index) + 1),
        output_dim=EMBED_DIM,
        input_length=MAX_LEN,
        mask_zero=True # <-- tells RNN to ignore padding (zeros)
    ),
    SimpleRNN(RNN_UNITS), # final hidden state only -> many-to-one
    Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
print("\nModel summary:")
model.summary()



Model summary:


/home/akashs/.local/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
E0000 00:00:1755602595.868517   26133 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1755602595.933853   26133 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [9]:
# !nvcc --version
# import tensorflow as tf
# print(tf.config.list_physical_devices('GPU'))


In [10]:

# -----------------------------
# 4) Train
# -----------------------------
history = model.fit(
    Xtr, ytr,
    epochs=8,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

# -----------------------------
# 5) Evaluate
# -----------------------------
loss, acc = model.evaluate(Xte, yte, verbose=0)
print(f"\nTest accuracy: {acc:.3f}")

Epoch 1/8
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8825 - loss: 0.3623 - val_accuracy: 0.9411 - val_loss: 0.1695
Epoch 2/8
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9699 - loss: 0.1031 - val_accuracy: 0.9679 - val_loss: 0.1000
Epoch 3/8
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9886 - loss: 0.0489 - val_accuracy: 0.9813 - val_loss: 0.0615
Epoch 4/8
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9953 - loss: 0.0246 - val_accuracy: 0.9839 - val_loss: 0.0570
Epoch 5/8
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9970 - loss: 0.0174 - val_accuracy: 0.9853 - val_loss: 0.0533
Epoch 6/8
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9990 - loss: 0.0093 - val_accuracy: 0.9826 - val_loss: 0.0596
Epoch 7/8
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9997 - loss: 0.0056 - val_accuracy: 0.9880 - val_loss: 0.0551
Epoch 8/8
94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9997 - loss: 0.0044 - val_accuracy: 0.9866 - val_loss: 0.0554


In [11]:

# -----------------------------
# 6) Predict on new sentences
# -----------------------------
def predict_sentiment(texts):
    if isinstance(texts, str):
        texts = [texts]
    cleaned = [clean_text(t) for t in texts]
    pad = to_padded(cleaned)
    probs = model.predict(pad, verbose=0).ravel()
    labels = (probs >= 0.5).astype(int)
    return list(zip(texts, probs, ["spam" if i==1 else "ham" for i in labels]))

examples = [
    "Congratulations! You have won a $1000 Walmart gift card. Click here to claim.",
    "Reminder: Your electricity bill payment is due tomorrow.",
    "Earn money fast working from home. Limited offer, sign up now!",
    "Meeting scheduled for 3 PM today. Please confirm your attendance.",
    "Get cheap medicines online without prescription. Order today!",
    "Your Amazon order has been shipped and will arrive soon.",
    "Exclusive deal just for you! 90% discount on luxury watches."
]

for txt, p, lab in predict_sentiment(examples):
    print(f"{lab:9s} | {p:.3f} | {txt}")

spam      | 0.991 | Congratulations! You have won a $1000 Walmart gift card. Click here to claim.
ham       | 0.002 | Reminder: Your electricity bill payment is due tomorrow.
ham       | 0.002 | Earn money fast working from home. Limited offer, sign up now!
ham       | 0.096 | Meeting scheduled for 3 PM today. Please confirm your attendance.
ham       | 0.003 | Get cheap medicines online without prescription. Order today!
spam      | 0.980 | Your Amazon order has been shipped and will arrive soon.
ham       | 0.003 | Exclusive deal just for you! 90% discount on luxury watches.


# END OF CODE